# Preprocessing on High Volume For-Hire Vehicle (HVFHV) Trip Records Dataset:

In this notebook, we are mainly focusing on convertint textual data to numerical data, and drop unused columns.

----

# Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql.functions import when, col
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("preprocessing_hvfhv")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

# Read Files:

Read Preprocessed HVFHV Parquet Files:

In [ ]:
base_dir = "../data"

In [ ]:
hvfhv_path = base_dir + '/curated/hvfhv_data/preprocessed_hvfhv'
hvfhv_sdf = spark.read.parquet(hvfhv_path)
hvfhv_sdf.show(5)

In [ ]:
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

In [ ]:
hvfhv_sdf.printSchema()

# Drop Unrelated Columns:

In [ ]:
hvfhv_sdf = hvfhv_sdf.drop("dispatching_base_num", "access_a_ride_flag")

num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

hvfhv_sdf.show(5)

# Data Encoding:

We do this because the correlation heatmap only accepts numerical data.

In [ ]:
num_license_num = hvfhv_sdf.select('hvfhs_license_num').distinct().count()
print(num_license_num)

Apply labeling encoding on `hvfhs_license_num`:

In [ ]:
unique_license_num = hvfhv_sdf.select('hvfhs_license_num').distinct()
unique_license_num.show(truncate=False)

In [ ]:
# Replace 'HV0003' with 0 and 'HV0005' with 1
hvfhv_sdf = hvfhv_sdf.withColumn('hvfhs_license_num', 
                                 when(col('hvfhs_license_num') == 'HV0005', 1).otherwise(0))

In [ ]:
hvfhv_sdf.show(5)

Apply labeling encoding on `shared_request_flag`, `shared_match_flag`, `wav_request_flag`, and `wav_match_flag`:

In [ ]:
# Replace 'N' with 0 and 'Y' with 1 in the following columns
columns_to_replace = ['shared_request_flag', 'shared_match_flag', 'wav_request_flag', 'wav_match_flag']

for column in columns_to_replace:
    hvfhv_sdf = hvfhv_sdf.withColumn(column, when(col(column) == 'Y', 1).otherwise(0))

In [ ]:
hvfhv_sdf.show(5)

Confirm all columns are numeric except datetime columns:

In [ ]:
hvfhv_sdf.printSchema()

# Save the Preprocessed HVFHV Dataset:

In [ ]:
hvfhv_dir = base_dir + '/curated/hvfhv_data'
file_name = 'preprocessed_hvfhv_2'
hvfhv_path = os.path.join(hvfhv_dir, file_name)
hvfhv_sdf.write.mode('overwrite').parquet(hvfhv_path)